In [2]:
# CELL 1: Import libraries
import pandas as pd
import numpy as np
import re
import os
from urllib.parse import urlparse



In [3]:
# Load raw dataset
df = pd.read_csv("../../data/raw/malicious_urls.csv")

print(f"Shape: {df.shape}")
print(f"\nLabel distribution:")
print(df["type"].value_counts())
print(f"\nSample:")
print(df.head(5).to_string())

Shape: (49750, 2)

Label distribution:
type
benign        36419
defacement     9158
phishing       3002
malware        1171
Name: count, dtype: int64

Sample:
                                                                                                                                                                                                        url        type
0                                                                                                                                                                                          br-icloud.com.br    phishing
1                                                                                                                                                                       mp3raid.com/music/krizz_kaliko.html      benign
2                                                                                                                                                                           bopsecrets.org/rexrot

In [5]:
# Check missing values & duplicates
print("Missing Values")
print(df.isnull().sum())

print(f"\nDuplicate URLs")
print(f"Total duplicates: {df.duplicated(subset='url').sum()}")

# Xóa duplicate, giữ lần đầu xuất hiện
df.drop_duplicates(subset="url", keep="first", inplace=True)
df.reset_index(drop=True, inplace=True)

print(f"Shape after removing duplicates: {df.shape}")

Missing Values
url     0
type    0
dtype: int64

Duplicate URLs
Total duplicates: 0
Shape after removing duplicates: (40439, 2)


In [6]:
# Encode labels (multi-class)
label_map = {
    "benign"    : 0,
    "defacement": 1,
    "phishing"  : 2,
    "malware"   : 3
}

df["label"] = df["type"].map(label_map)

print("Label mapping:", label_map)
print("\nDistribution after encoding:")
print(df["label"].value_counts().sort_index())

Label mapping: {'benign': 0, 'defacement': 1, 'phishing': 2, 'malware': 3}

Distribution after encoding:
label
0    29854
1     7455
2     2459
3      671
Name: count, dtype: int64


In [7]:
# Extract URL structural features
def has_ip_address(url):
    # check if URL contains an IP address instead of domain name
    ip_pattern = re.compile(
        r"(\d{1,3}\.){3}\d{1,3}"
    )
    return int(bool(ip_pattern.search(url)))

def extract_url_features(url):
    url = str(url)
    
    try:
        parsed = urlparse(url if url.startswith("http") 
                         else "http://" + url)
        domain = parsed.netloc or parsed.path.split("/")[0]
        path   = parsed.path
        query  = parsed.query
    except:
        domain, path, query = "", "", ""

    return {
        "url_length"       : len(url),
        "domain_length"    : len(domain),
        "path_length"      : len(path),
        "num_dots"         : url.count("."),
        "num_slashes"      : url.count("/"),
        "num_digits"       : sum(c.isdigit() for c in url),
        "num_special_chars": len(re.findall(r"[=?&@%#_~]", url)),
        "num_subdomains"   : max(0, domain.count(".") - 1),
        "has_https"        : int(url.startswith("https")),
        "has_ip"           : has_ip_address(url),
        "has_at_symbol"    : int("@" in url),   # trick redirect
        "has_double_slash" : int("//" in url[7:]),  # ignore http://
        "query_length"     : len(query),
        "num_query_params" : len(query.split("&")) if query else 0,
    }

# Apply to the entire dataset
features_df = df["url"].apply(extract_url_features).apply(pd.Series)

print(f"Features extracted: {features_df.shape[1]} features")
print(features_df.head(3).to_string())

Features extracted: 14 features
   url_length  domain_length  path_length  num_dots  num_slashes  num_digits  num_special_chars  num_subdomains  has_https  has_ip  has_at_symbol  has_double_slash  query_length  num_query_params
0          16             16            0         2            0           0                  0               1          0       0              0                 0             0                 0
1          35             11           24         2            2           1                  1               0          0       0              0                 0             0                 0
2          31             14           17         2            3           1                  0               0          0       0              0                 0             0                 0


In [8]:
# Merge features with labels
df_processed = pd.concat(
    [df[["url", "type", "label"]].reset_index(drop=True),
     features_df.reset_index(drop=True)],
    axis=1
)

print(f"Final shape: {df_processed.shape}")
print(f"\nColumns: {df_processed.columns.tolist()}")
print(f"\nSample:")
print(df_processed.head(3).to_string())

Final shape: (40439, 17)

Columns: ['url', 'type', 'label', 'url_length', 'domain_length', 'path_length', 'num_dots', 'num_slashes', 'num_digits', 'num_special_chars', 'num_subdomains', 'has_https', 'has_ip', 'has_at_symbol', 'has_double_slash', 'query_length', 'num_query_params']

Sample:
                                   url      type  label  url_length  domain_length  path_length  num_dots  num_slashes  num_digits  num_special_chars  num_subdomains  has_https  has_ip  has_at_symbol  has_double_slash  query_length  num_query_params
0                     br-icloud.com.br  phishing      2          16             16            0         2            0           0                  0               1          0       0              0                 0             0                 0
1  mp3raid.com/music/krizz_kaliko.html    benign      0          35             11           24         2            2           1                  1               0          0       0              0          

In [9]:
# Feature statistics by URL type
feature_cols = features_df.columns.tolist()

stats = df_processed.groupby("type")[
    ["url_length", "num_dots", "has_https", 
     "has_ip", "num_special_chars"]
].mean().round(3)

print("Average Feature Values by URL Type")
print(stats.to_string())

Average Feature Values by URL Type
            url_length  num_dots  has_https  has_ip  num_special_chars
type                                                                  
benign          53.100     1.685      0.005   0.000              1.492
defacement      86.151     2.840      0.000   0.000              5.186
malware         75.808     2.180      0.006   0.006              8.724
phishing        41.939     2.312      0.006   0.006              0.442


In [11]:
# Export processed dataset
os.makedirs("../../data/processed", exist_ok=True)

output_path = "../../data/processed/urls_processed.csv"
df_processed.to_csv(output_path, index=False, encoding="utf-8")

print(f"Saved → {output_path}")
print(f"Shape : {df_processed.shape}")

Saved → ../../data/processed/urls_processed.csv
Shape : (40439, 17)
